In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="05-harmful-content-detection",
    notebook_name="01_harmful_content_system_design.ipynb"
)

# Harmful Content Detection: System Design

## What We Are Building

Imagine you are the **hall monitor for the entire internet**. Every second, millions of people post photos, videos, memes, and messages. Your job? Spot the mean, dangerous, or inappropriate stuff *before* anyone gets hurt -- and do it almost instantly.

That is what a **harmful content detection system** does. Think of it like an airport security scanner, but for social media posts. Every post goes through the scanner before it reaches other users. If the scanner is highly confident the post is dangerous, it gets removed immediately. If it is only slightly suspicious, it gets set aside for a human inspector to check.

In this notebook, we design the complete system from scratch:

1. Problem clarification (interview dialogue)
2. Framing as an ML task
3. Handling multi-modal data (text + images + video)
4. Early vs. Late fusion strategies
5. ML category selection (why multi-task wins)
6. Data pipeline design
7. Complete system architecture
8. Interview cheat sheet

---

## 1. Problem Clarification (Interview Dialogue)

In a real interview, ALWAYS start by asking clarifying questions. Here is how the conversation typically goes:

| # | Candidate Question | Interviewer Answer | Why It Matters |
|---|-------------------|-------------------|----------------|
| 1 | Does the system detect harmful **content** or **bad actors**? | Both matter, but focus on harmful content | Scopes the problem -- bad actors (fake accounts, spam) need different approaches |
| 2 | What modalities can a post contain? | Text, image, video, or any combination | Determines model architecture (single-modal vs. multi-modal) |
| 3 | What languages are supported? | Multiple languages | Need multilingual models (DistilmBERT, not English-only BERT) |
| 4 | Which harm categories? (violence, nudity, hate speech...) | Major ones; skip misinformation for now | Each category becomes a classification task |
| 5 | Are human annotators available? | Yes, but limited (~10K labels/day with 500M posts/day) | Cannot label everything manually -- need smart labeling strategy |
| 6 | Can users report harmful posts? | Yes | User reports become a free labeling signal (natural labels) |
| 7 | Should we explain WHY a post was removed? | Yes, essential for user trust | Rules out single binary classifier |
| 8 | Latency requirements? | Varies -- violence needs real-time, others can be batch | Architectural decision: sync vs. async processing |
| 9 | What happens to flagged posts? | Either removed or demoted based on confidence | Need calibrated probability outputs, not just yes/no |

### The Two Categories of Integrity Enforcement

| Category | Examples | Our Focus? |
|----------|----------|------------|
| **Harmful content** | Violence, nudity, self-harm, hate speech | YES -- this notebook |
| **Bad acts / bad actors** | Fake accounts, spam, phishing, organized manipulation | No (different system design) |

**Key interview phrase**: *"I will focus on harmful content detection specifically, since bad actor detection is a separate system that relies more on behavioral patterns than content analysis."*

In [ ]:
# Let's make the problem statement concrete with numbers

problem_scope = {
    'daily_posts': 500_000_000,          # 500 million posts/day
    'human_labels_per_day': 10_000,      # Expensive human annotators
    'harm_categories': ['Violence', 'Nudity', 'Self-harm', 'Hate Speech', 'Harassment'],
    'modalities': ['Text', 'Image', 'Video'],
    'languages': 'Multilingual',
    'latency_requirements': {
        'violence': 'Real-time (seconds)',
        'nudity': 'Near real-time (minutes)',
        'hate_speech': 'Can tolerate slight delay',
    },
    'actions': ['Remove immediately', 'Demote visibility', 'Queue for human review']
}

print("=== Problem Scope ===")
for key, value in problem_scope.items():
    if isinstance(value, dict):
        print(f"\n  {key}:")
        for k, v in value.items():
            print(f"    {k}: {v}")
    elif isinstance(value, list):
        print(f"  {key}: {', '.join(str(v) for v in value)}")
    else:
        print(f"  {key}: {value:,}" if isinstance(value, int) else f"  {key}: {value}")

# The labeling gap is HUGE
labeling_coverage = problem_scope['human_labels_per_day'] / problem_scope['daily_posts'] * 100
print(f"\nHuman labeling covers only {labeling_coverage:.4f}% of daily posts!")
print("This is why we need ML -- humans alone cannot keep up.")

---

## 2. Frame as an ML Task

### ML Objective

**Accurately predict the probability that a post is harmful, and in which category.**

Think of it like a doctor's diagnosis. The doctor does not just say "you are sick" -- they tell you WHAT you have (flu, cold, infection). Similarly, our system does not just say "harmful" -- it says "harmful because of violence" or "harmful because of hate speech."

### Input / Output

```
Input:  A post P (text + image + video + metadata)
Output: Probability scores for EACH harmful class
        e.g., P(violence) = 0.92, P(nudity) = 0.01, P(hate_speech) = 0.15, ...
```

### Why Probabilities Instead of Yes/No?

Because we need **calibrated confidence**:
- P(violence) = 0.99 --> Remove immediately (high confidence)
- P(violence) = 0.65 --> Demote and queue for human review (uncertain)
- P(violence) = 0.02 --> Let it through (low risk)

The threshold can be tuned per category based on the cost of mistakes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Simulate what the model outputs for a single post
np.random.seed(42)

# Example predictions for 3 different posts
categories = ['Violence', 'Nudity', 'Self-harm', 'Hate Speech', 'Harassment']

posts = {
    'Post A: Hateful meme': [0.15, 0.03, 0.02, 0.91, 0.45],
    'Post B: Violent video': [0.97, 0.12, 0.08, 0.05, 0.03],
    'Post C: Normal photo': [0.01, 0.02, 0.00, 0.01, 0.03],
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#e74c3c', '#e67e22', '#9b59b6', '#2980b9', '#27ae60']

for ax, (post_name, probs) in zip(axes, posts.items()):
    bars = ax.barh(categories, probs, color=colors, edgecolor='white', linewidth=1.5)
    ax.set_xlim(0, 1.0)
    ax.set_xlabel('Probability', fontsize=11)
    ax.set_title(post_name, fontsize=13, fontweight='bold')
    
    # Add threshold lines
    ax.axvline(x=0.8, color='red', linestyle='--', alpha=0.7, label='Remove threshold')
    ax.axvline(x=0.4, color='orange', linestyle='--', alpha=0.7, label='Demote threshold')
    
    # Add probability labels
    for bar, prob in zip(bars, probs):
        ax.text(prob + 0.02, bar.get_y() + bar.get_height()/2, 
                f'{prob:.2f}', va='center', fontsize=10, fontweight='bold')
    
    if ax == axes[0]:
        ax.legend(fontsize=8, loc='lower right')

plt.suptitle('Model Output: Per-Category Harm Probabilities', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Post A: Hate speech detected (0.91 > 0.8 threshold) --> REMOVE immediately")
print("Post B: Violence detected (0.97 > 0.8 threshold) --> REMOVE immediately")
print("Post C: All probabilities below 0.4 --> Let it through normally")

---

## 3. Handling Multi-Modal Data

Here is the tricky part. A post is not just text. It can be:
- **Just text**: "I hate everyone in this group"
- **Just an image**: A violent photo
- **Text + Image (meme)**: A harmless image + harmless text = harmful together!
- **Video**: Contains violent or inappropriate scenes
- **Any combination**: Text + image + video + metadata

Think of it like a **report card**. To understand how a student is doing, you cannot just look at one subject -- you need ALL subjects to get the full picture.

### The Meme Problem

This is the killer insight that separates good answers from great ones in interviews:

```
Image alone: Photo of a frog  --> Harmless
Text alone:  "Me when Monday" --> Harmless  
Combined:    Frog + "Me when I see minorities" --> HARMFUL (hate speech)
```

The harm only emerges from the **combination** of modalities. This is why our fusion strategy matters enormously.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ============================================================
# LEFT: Late Fusion
# ============================================================
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_title('Late Fusion\n(Separate Models, Combined at End)', fontsize=14, fontweight='bold', pad=10)
ax.axis('off')

# Input modalities
for y, (label, color) in zip([6, 4, 2], 
    [('Image', '#FFCDD2'), ('Text', '#BBDEFB'), ('Author\nMetadata', '#C8E6C9')]):
    box = FancyBboxPatch((0.2, y-0.4), 1.8, 0.8, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(1.1, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Individual models
for y, label in zip([6, 4, 2], ['Image\nModel', 'Text\nModel', 'Metadata\nModel']):
    box = FancyBboxPatch((3.2, y-0.4), 1.8, 0.8, boxstyle='round,pad=0.1',
                         facecolor='#FFF9C4', edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(4.1, y, label, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.annotate('', xy=(3.2, y), xytext=(2.0, y),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Individual predictions
for y, label in zip([6, 4, 2], ['pred_1', 'pred_2', 'pred_3']):
    ax.text(6.0, y, label, ha='center', va='center', fontsize=10, fontweight='bold', color='#666')
    ax.annotate('', xy=(5.6, y), xytext=(5.0, y),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Combine
box = FancyBboxPatch((6.8, 3.2), 1.8, 1.6, boxstyle='round,pad=0.1',
                     facecolor='#E1BEE7', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(7.7, 4.0, 'Combine\nPredictions', ha='center', va='center', fontsize=10, fontweight='bold')

for y in [6, 4, 2]:
    ax.annotate('', xy=(6.8, 4.0), xytext=(6.4, y),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))

# Final prediction
box = FancyBboxPatch((8.2, 0.5), 1.5, 0.8, boxstyle='round,pad=0.1',
                     facecolor='#FFAB91', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(8.95, 0.9, 'Final\nPrediction', ha='center', va='center', fontsize=10, fontweight='bold')
ax.annotate('', xy=(8.95, 1.3), xytext=(7.7, 3.2),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Problem callout
ax.text(5.0, 0.5, 'PROBLEM: Cannot\ndetect harmful memes!', ha='center', va='center',
        fontsize=11, fontweight='bold', color='red',
        bbox=dict(boxstyle='round', facecolor='#FFEBEE', alpha=0.9))

# ============================================================
# RIGHT: Early Fusion
# ============================================================
ax = axes[1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_title('Early Fusion (Our Choice)\n(Combine Features, Then One Model)', fontsize=14, fontweight='bold', pad=10)
ax.axis('off')

# Input modalities
for y, (label, color) in zip([6, 4, 2], 
    [('Image', '#FFCDD2'), ('Text', '#BBDEFB'), ('Author\nMetadata', '#C8E6C9')]):
    box = FancyBboxPatch((0.2, y-0.4), 1.8, 0.8, boxstyle='round,pad=0.1',
                         facecolor=color, edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(1.1, y, label, ha='center', va='center', fontsize=10, fontweight='bold')

# Feature extractors
for y, label in zip([6, 4, 2], ['Image\nEncoder', 'Text\nEncoder', 'Feature\nEncoder']):
    box = FancyBboxPatch((3.0, y-0.4), 1.6, 0.8, boxstyle='round,pad=0.1',
                         facecolor='#B3E5FC', edgecolor='#333', linewidth=2)
    ax.add_patch(box)
    ax.text(3.8, y, label, ha='center', va='center', fontsize=9, fontweight='bold')
    ax.annotate('', xy=(3.0, y), xytext=(2.0, y),
                arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Feature fusion
box = FancyBboxPatch((5.5, 3.0), 1.6, 2.0, boxstyle='round,pad=0.1',
                     facecolor='#FFF9C4', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(6.3, 4.0, 'Feature\nFusion\n(Concatenate)', ha='center', va='center', fontsize=9, fontweight='bold')

for y in [6, 4, 2]:
    ax.annotate('', xy=(5.5, 4.0), xytext=(4.6, y),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))

# Single model
box = FancyBboxPatch((7.8, 3.0), 1.8, 2.0, boxstyle='round,pad=0.1',
                     facecolor='#C8E6C9', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(8.7, 4.0, 'Single\nML Model', ha='center', va='center', fontsize=11, fontweight='bold')
ax.annotate('', xy=(7.8, 4.0), xytext=(7.1, 4.0),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Final prediction
box = FancyBboxPatch((8.0, 0.5), 1.5, 0.8, boxstyle='round,pad=0.1',
                     facecolor='#FFAB91', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(8.75, 0.9, 'Final\nPrediction', ha='center', va='center', fontsize=10, fontweight='bold')
ax.annotate('', xy=(8.7, 1.3), xytext=(8.7, 3.0),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Success callout
ax.text(4.5, 0.5, 'CAN detect harmful\nmemes! (cross-modal)', ha='center', va='center',
        fontsize=11, fontweight='bold', color='green',
        bbox=dict(boxstyle='round', facecolor='#E8F5E9', alpha=0.9))

plt.tight_layout()
plt.show()

In [ ]:
# Let's build a comparison table in code
import pandas as pd

fusion_comparison = pd.DataFrame({
    'Criterion': [
        'Cross-modal detection (memes)',
        'Training data requirements',
        'Independent model improvement',
        'Debugging ease',
        'Adding new modalities',
        'Number of models to maintain',
        'Training complexity',
        'Overall recommendation'
    ],
    'Late Fusion': [
        'FAILS -- each model sees only its modality',
        'Need separate datasets per modality',
        'Easy -- update one model at a time',
        'Easy -- isolate which model is wrong',
        'Easy -- just add another model',
        'Multiple (one per modality)',
        'Simpler per model',
        'Use when modalities are independent'
    ],
    'Early Fusion': [
        'YES -- model sees all modalities together',
        'One unified dataset',
        'Harder -- everything is coupled',
        'Harder -- entangled features',
        'Requires retraining',
        'One single model',
        'More complex but one-time effort',
        'USE THIS -- captures cross-modal harm'
    ]
})

print("=== Late Fusion vs. Early Fusion ===")
print(fusion_comparison.to_string(index=False))
print("\nOUR CHOICE: Early Fusion")
print("REASON: Memes and combined content can be harmful even when each modality")
print("        is benign in isolation. With 500M posts/day, we have enough data")
print("        for the model to learn these cross-modal relationships.")

---

## 4. ML Category: Why Multi-Task Classification Wins

Now that we have decided on early fusion for combining modalities, we need to choose HOW to structure the classification. Think of it like organizing a school with different classrooms:

### Option A: Single Binary Classifier
One teacher who just says "PASS" or "FAIL" for the whole school.
- **Problem**: Cannot tell you WHICH subject you failed. Cannot explain why a post was removed.

### Option B: One Binary Classifier Per Class  
Separate teachers for math, science, English, etc.
- **Pro**: Can explain which subject failed.
- **Con**: Training and maintaining separate teachers is expensive.

### Option C: Multi-Label Classifier
One teacher grades everything using the same rubric.
- **Pro**: Cheaper (one model).
- **Con**: Different harm types might need different "rubrics" (feature transformations).

### Option D: Multi-Task Classifier (Our Winner!)
One school (shared layers) with specialized departments (task-specific heads).
- **Pro**: Shared knowledge + specialized expertise. Best of all worlds.
- Think of it like a hospital: all doctors share the same MRI machine and blood tests (shared layers), but the cardiologist interprets heart results differently than the neurologist interprets brain results (task-specific heads).

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# ============================================================
# Option A: Single Binary Classifier
# ============================================================
ax = axes[0, 0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.set_title('Option A: Single Binary Classifier', fontsize=13, fontweight='bold')
ax.axis('off')

box = FancyBboxPatch((0.5, 2.0), 2.5, 2.0, boxstyle='round,pad=0.15',
                     facecolor='#B3E5FC', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(1.75, 3.0, 'Fused\nFeatures', ha='center', va='center', fontsize=11, fontweight='bold')

box = FancyBboxPatch((4.5, 2.0), 2.5, 2.0, boxstyle='round,pad=0.15',
                     facecolor='#FFF9C4', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(5.75, 3.0, 'Single\nModel', ha='center', va='center', fontsize=11, fontweight='bold')

box = FancyBboxPatch((8.0, 2.3), 1.5, 1.4, boxstyle='round,pad=0.1',
                     facecolor='#FFCDD2', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(8.75, 3.0, 'Harmful?\nYes / No', ha='center', va='center', fontsize=10, fontweight='bold')

ax.annotate('', xy=(4.5, 3.0), xytext=(3.0, 3.0),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))
ax.annotate('', xy=(8.0, 3.0), xytext=(7.0, 3.0),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

ax.text(5.0, 0.5, 'Problem: Cannot explain WHY post was removed', 
        ha='center', fontsize=11, color='red', fontweight='bold')

# ============================================================
# Option B: One Binary Classifier Per Class
# ============================================================
ax = axes[0, 1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.set_title('Option B: One Classifier Per Class', fontsize=13, fontweight='bold')
ax.axis('off')

box = FancyBboxPatch((0.3, 1.5), 2.0, 3.0, boxstyle='round,pad=0.15',
                     facecolor='#B3E5FC', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(1.3, 3.0, 'Fused\nFeatures', ha='center', va='center', fontsize=11, fontweight='bold')

classes = ['Violence', 'Nudity', 'Hate Speech']
colors_b = ['#FFCDD2', '#F8BBD0', '#E1BEE7']
for i, (cls, c) in enumerate(zip(classes, colors_b)):
    y = 4.5 - i * 1.5
    box = FancyBboxPatch((4.0, y-0.35), 2.0, 0.7, boxstyle='round,pad=0.1',
                         facecolor='#FFF9C4', edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(5.0, y, f'{cls}\nModel', ha='center', va='center', fontsize=9, fontweight='bold')
    
    box = FancyBboxPatch((7.5, y-0.25), 1.8, 0.5, boxstyle='round,pad=0.1',
                         facecolor=c, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(8.4, y, f'P({cls[:4]})', ha='center', va='center', fontsize=9, fontweight='bold')
    
    ax.annotate('', xy=(4.0, y), xytext=(2.3, 3.0),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))
    ax.annotate('', xy=(7.5, y), xytext=(6.0, y),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

ax.text(5.0, 0.3, 'Problem: N separate models to train & maintain!', 
        ha='center', fontsize=11, color='red', fontweight='bold')

# ============================================================
# Option C: Multi-Label Classifier
# ============================================================
ax = axes[1, 0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.set_title('Option C: Multi-Label Classifier', fontsize=13, fontweight='bold')
ax.axis('off')

box = FancyBboxPatch((0.3, 1.5), 2.0, 3.0, boxstyle='round,pad=0.15',
                     facecolor='#B3E5FC', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(1.3, 3.0, 'Fused\nFeatures', ha='center', va='center', fontsize=11, fontweight='bold')

box = FancyBboxPatch((3.8, 1.5), 2.5, 3.0, boxstyle='round,pad=0.15',
                     facecolor='#FFF9C4', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(5.05, 3.0, 'Single Shared\nModel\n(All weights\nshared)', ha='center', va='center', fontsize=10, fontweight='bold')

ax.annotate('', xy=(3.8, 3.0), xytext=(2.3, 3.0),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

for i, (cls, c) in enumerate(zip(classes, colors_b)):
    y = 4.5 - i * 1.5
    box = FancyBboxPatch((7.5, y-0.25), 1.8, 0.5, boxstyle='round,pad=0.1',
                         facecolor=c, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(8.4, y, f'P({cls[:4]})', ha='center', va='center', fontsize=9, fontweight='bold')
    ax.annotate('', xy=(7.5, y), xytext=(6.3, 3.0),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))

ax.text(5.0, 0.3, 'Problem: Same feature transform for different harm types', 
        ha='center', fontsize=11, color='red', fontweight='bold')

# ============================================================
# Option D: Multi-Task Classifier (OUR CHOICE)
# ============================================================
ax = axes[1, 1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.set_title('Option D: Multi-Task Classifier (OUR CHOICE)', fontsize=13, fontweight='bold', color='green')
ax.axis('off')

# Fused features
box = FancyBboxPatch((0.2, 1.5), 1.6, 3.0, boxstyle='round,pad=0.15',
                     facecolor='#B3E5FC', edgecolor='#333', linewidth=2)
ax.add_patch(box)
ax.text(1.0, 3.0, 'Fused\nFeatures', ha='center', va='center', fontsize=10, fontweight='bold')

# Shared layers
box = FancyBboxPatch((2.5, 1.5), 2.0, 3.0, boxstyle='round,pad=0.15',
                     facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2.5)
ax.add_patch(box)
ax.text(3.5, 3.0, 'Shared\nLayers\n(common\npatterns)', ha='center', va='center', fontsize=9, fontweight='bold')
ax.annotate('', xy=(2.5, 3.0), xytext=(1.8, 3.0),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# Task-specific heads
for i, (cls, c) in enumerate(zip(classes, ['#FFCDD2', '#F8BBD0', '#E1BEE7'])):
    y = 4.5 - i * 1.5
    box = FancyBboxPatch((5.5, y-0.35), 1.8, 0.7, boxstyle='round,pad=0.1',
                         facecolor=c, edgecolor='#333', linewidth=1.5)
    ax.add_patch(box)
    ax.text(6.4, y, f'{cls}\nHead', ha='center', va='center', fontsize=9, fontweight='bold')
    ax.annotate('', xy=(5.5, y), xytext=(4.5, 3.0),
                arrowprops=dict(arrowstyle='->', color='#666', lw=1.5))
    
    # Outputs
    box = FancyBboxPatch((8.2, y-0.25), 1.5, 0.5, boxstyle='round,pad=0.1',
                         facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=1.5)
    ax.add_patch(box)
    ax.text(8.95, y, f'P({cls[:4]})', ha='center', va='center', fontsize=9, fontweight='bold')
    ax.annotate('', xy=(8.2, y), xytext=(7.3, y),
                arrowprops=dict(arrowstyle='->', color='#333', lw=1.5))

ax.text(5.0, 0.3, 'BEST: Shared knowledge + Specialized expertise!', 
        ha='center', fontsize=11, color='green', fontweight='bold')

plt.suptitle('Four Ways to Structure Harmful Content Classification', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Why multi-task beats the alternatives -- quantified comparison
import pandas as pd

comparison = pd.DataFrame({
    'Criterion': [
        'Can explain removal reason?',
        'Number of models to maintain',
        'Knowledge sharing across tasks',
        'Works with limited per-task data',
        'Specialized per harm type',
        'Training cost',
        'Serving cost',
    ],
    'A: Single Binary': [
        'NO', '1', 'N/A', 'N/A', 'NO', 'Low', 'Low',
    ],
    'B: Per-Class Binary': [
        'YES', 'N (one per class)', 'NO', 'NO', 'YES', 'HIGH (N models)', 'HIGH',
    ],
    'C: Multi-Label': [
        'YES', '1', 'Partial (shared weights)', 'Partial', 'NO (same transform)', 'Low', 'Low',
    ],
    'D: Multi-Task': [
        'YES', '1', 'YES (shared layers)', 'YES (data sharing)', 'YES (task heads)', 'Medium', 'Low',
    ],
})

print("=== ML Category Comparison ===")
print(comparison.to_string(index=False))
print()
print("WINNER: Multi-Task Classifier (Option D)")
print()
print("Three key advantages of multi-task learning:")
print("1. EFFICIENT: One model to train and maintain (vs. N separate models)")
print("2. SMART SHARING: Shared layers learn patterns useful for ALL tasks")
print("   (e.g., understanding violent imagery helps detect both violence AND self-harm)")
print("3. DATA SHARING: Training data for violence helps the shared layers learn")
print("   patterns that also help hate speech detection (transfer learning across tasks)")

---

## 5. Data Pipeline

Every ML system needs data. Ours comes from three main sources. Think of it like a restaurant:
- **Users table** = Your customer database (who are they?)
- **Posts table** = The food being served (what is the content?)
- **Interactions table** = Customer feedback cards (how did people react?)

In [ ]:
import pandas as pd
import numpy as np

# ======================================================================
# Data Source 1: Users Table
# ======================================================================
users_df = pd.DataFrame({
    'user_id': [1, 2, 3, 4, 5],
    'username': ['alice_wonder', 'bob_builder', 'charlie_d', 'dana_k', 'eve_online'],
    'age': [28, 45, 16, 32, 22],
    'gender': ['F', 'M', 'M', 'F', 'F'],
    'city': ['San Francisco', 'London', 'Tokyo', 'Mumbai', 'Berlin'],
    'country': ['USA', 'UK', 'Japan', 'India', 'Germany'],
    'account_age_days': [1825, 3650, 30, 730, 365],
    'num_followers': [1200, 5000, 50, 800, 15000],
    'num_violations': [0, 0, 3, 0, 1],
})

print("=== Users Table ===")
print(users_df.to_string(index=False))
print(f"\nNotice: user 3 (charlie_d) has 3 violations and a 30-day-old account.")
print("This is a strong signal for the model -- repeat offenders are more likely to post harmful content.")
print()

In [ ]:
# ======================================================================
# Data Source 2: Posts Table
# ======================================================================
posts_df = pd.DataFrame({
    'post_id': [101, 102, 103, 104, 105],
    'author_id': [1, 3, 2, 3, 5],
    'timestamp': pd.to_datetime([
        '2024-01-15 08:30:00', '2024-01-15 14:22:00',
        '2024-01-15 19:45:00', '2024-01-15 23:10:00',
        '2024-01-16 06:00:00'
    ]),
    'text_content': [
        'Starting my diet today! Wish me luck!',
        'Check out this crazy fight video',
        'The sunset was beautiful tonight',
        'I hate everyone in this group',
        'New recipe for chocolate cake'
    ],
    'has_image': [True, False, True, True, True],
    'has_video': [False, True, False, False, False],
    'device': ['smartphone', 'smartphone', 'desktop', 'smartphone', 'tablet'],
})

print("=== Posts Table ===")
print(posts_df.to_string(index=False))
print(f"\nPosts 102 and 104 are from user 3 (the repeat offender).")
print("Post 102 mentions 'fight video' -- likely violent content.")
print("Post 104 says 'I hate everyone' -- potential hate speech / harassment.")
print()

In [ ]:
# ======================================================================
# Data Source 3: User-Post Interactions Table
# ======================================================================
interactions_df = pd.DataFrame({
    'user_id': [2, 4, 5, 1, 2, 4, 5, 1],
    'post_id': [102, 102, 102, 104, 104, 104, 104, 103],
    'interaction_type': [
        'Impression', 'Comment', 'Report',
        'Report', 'Comment', 'Report', 'Report',
        'Like'
    ],
    'interaction_content': [
        None, 'This is disturbing!', 'violence',
        'hate speech', 'This is unacceptable', 'harassment', 'hate speech',
        None
    ],
})

print("=== User-Post Interactions Table ===")
print(interactions_df.to_string(index=False))
print()

# Aggregate signals
print("=== Aggregated Signals Per Post ===")
for post_id in [102, 103, 104]:
    post_interactions = interactions_df[interactions_df['post_id'] == post_id]
    n_reports = len(post_interactions[post_interactions['interaction_type'] == 'Report'])
    n_comments = len(post_interactions[post_interactions['interaction_type'] == 'Comment'])
    n_likes = len(post_interactions[post_interactions['interaction_type'] == 'Like'])
    print(f"  Post {post_id}: {n_reports} reports, {n_comments} comments, {n_likes} likes")

print("\nPosts 102 and 104 have multiple reports -- strong harmful signal!")
print("Post 103 has only a like -- likely safe content.")
print("\nThese interaction signals are 'natural labels' -- free training data!")

---

## 6. System Architecture

Now let's put it all together. The complete system has three main services:

1. **Harmful Content Detection Service**: The ML model that predicts harm probability
2. **Violation Enforcement Service**: Immediately removes high-confidence harmful posts
3. **Demoting Service**: Reduces visibility of uncertain posts and queues for human review

Think of it like a three-tier court system:
- **Detection Service** = The investigation (gathering evidence, analyzing the case)
- **Violation Enforcement** = The judge who issues immediate sentences for clear violations
- **Demoting Service** = The appeals court that handles ambiguous cases more carefully

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(1, 1, figsize=(18, 12))
ax.set_xlim(0, 18)
ax.set_ylim(0, 13)
ax.axis('off')
ax.set_title('Harmful Content Detection System Architecture', fontsize=18, fontweight='bold', pad=20)

# ---- User creates post ----
box = FancyBboxPatch((0.5, 10.5), 3.0, 1.5, boxstyle='round,pad=0.15',
                     facecolor='#BBDEFB', edgecolor='#1565C0', linewidth=2.5)
ax.add_patch(box)
ax.text(2.0, 11.25, 'User Creates Post\n(text + image + video)', 
        ha='center', va='center', fontsize=11, fontweight='bold')

# ---- Feature Extraction Pipeline ----
box = FancyBboxPatch((5.0, 10.5), 4.0, 1.5, boxstyle='round,pad=0.15',
                     facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2.5)
ax.add_patch(box)
ax.text(7.0, 11.25, 'Feature Extraction Pipeline\nText (DistilBERT) + Image (CLIP)\n+ Author + Context + Reactions', 
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.annotate('', xy=(5.0, 11.25), xytext=(3.5, 11.25),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))

# ---- ML Model (multi-task) ----
box = FancyBboxPatch((5.5, 7.5), 3.0, 2.0, boxstyle='round,pad=0.15',
                     facecolor='#FFF9C4', edgecolor='#F57F17', linewidth=2.5)
ax.add_patch(box)
ax.text(7.0, 8.5, 'Multi-Task\nML Model\n(Shared + Task Heads)', 
        ha='center', va='center', fontsize=11, fontweight='bold')
ax.annotate('', xy=(7.0, 9.5), xytext=(7.0, 10.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))

# ---- Prediction outputs ----
categories_out = ['P(violence)=0.97', 'P(nudity)=0.03', 'P(hate)=0.12', 'P(harm)=0.01']
for i, cat in enumerate(categories_out):
    ax.text(10.5, 9.0 - i * 0.5, cat, fontsize=9, fontweight='bold',
            fontfamily='monospace', color='#333',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='#F5F5F5', edgecolor='#999'))
ax.annotate('', xy=(10.3, 8.5), xytext=(8.5, 8.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2))

# ---- Decision Logic ----
# High confidence path
box = FancyBboxPatch((5.0, 5.0), 4.0, 1.5, boxstyle='round,pad=0.15',
                     facecolor='#FFECB3', edgecolor='#FF8F00', linewidth=2)
ax.add_patch(box)
ax.text(7.0, 5.75, 'Confidence\nThreshold Check', 
        ha='center', va='center', fontsize=11, fontweight='bold')
ax.annotate('', xy=(7.0, 6.5), xytext=(7.0, 7.5),
            arrowprops=dict(arrowstyle='->', color='#333', lw=2.5))

# ---- HIGH confidence --> Violation Enforcement ----
box = FancyBboxPatch((0.5, 1.5), 4.5, 2.5, boxstyle='round,pad=0.15',
                     facecolor='#FFCDD2', edgecolor='#C62828', linewidth=2.5)
ax.add_patch(box)
ax.text(2.75, 2.75, 'Violation Enforcement\nService\n\n- Immediately remove post\n- Notify user WHY\n- Log for analytics', 
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.annotate('', xy=(4.0, 4.0), xytext=(5.5, 5.0),
            arrowprops=dict(arrowstyle='->', color='#C62828', lw=2.5))
ax.text(3.5, 4.5, 'HIGH confidence', fontsize=10, fontweight='bold', color='#C62828')

# ---- LOW confidence --> Demoting Service ----
box = FancyBboxPatch((10.5, 1.5), 4.5, 2.5, boxstyle='round,pad=0.15',
                     facecolor='#FFE0B2', edgecolor='#E65100', linewidth=2.5)
ax.add_patch(box)
ax.text(12.75, 2.75, 'Demoting Service\n\n- Reduce post visibility\n- Queue for human review\n- Store for inspection', 
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.annotate('', xy=(11.5, 4.0), xytext=(8.5, 5.0),
            arrowprops=dict(arrowstyle='->', color='#E65100', lw=2.5))
ax.text(10.2, 4.5, 'LOW confidence', fontsize=10, fontweight='bold', color='#E65100')

# ---- NOT harmful --> Post goes live ----
box = FancyBboxPatch((13.5, 5.0), 3.5, 1.5, boxstyle='round,pad=0.15',
                     facecolor='#C8E6C9', edgecolor='#2E7D32', linewidth=2.5)
ax.add_patch(box)
ax.text(15.25, 5.75, 'Post Goes Live\nNormally', 
        ha='center', va='center', fontsize=11, fontweight='bold', color='#2E7D32')
ax.annotate('', xy=(13.5, 5.75), xytext=(9.0, 5.75),
            arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=2.5))
ax.text(11.0, 6.2, 'NOT harmful', fontsize=10, fontweight='bold', color='#2E7D32')

# ---- Human Review loop ----
box = FancyBboxPatch((11.0, 0.0), 3.5, 1.0, boxstyle='round,pad=0.1',
                     facecolor='#E1BEE7', edgecolor='#6A1B9A', linewidth=2)
ax.add_patch(box)
ax.text(12.75, 0.5, 'Human Review Queue', ha='center', va='center', fontsize=10, fontweight='bold')
ax.annotate('', xy=(12.75, 1.0), xytext=(12.75, 1.5),
            arrowprops=dict(arrowstyle='->', color='#6A1B9A', lw=1.5, ls='--'))

# ---- User Reports feedback loop ----
box = FancyBboxPatch((14.5, 7.5), 3.0, 2.0, boxstyle='round,pad=0.15',
                     facecolor='#E8EAF6', edgecolor='#283593', linewidth=2)
ax.add_patch(box)
ax.text(16.0, 8.5, 'User Reports\n& Appeals\n(feedback loop)', 
        ha='center', va='center', fontsize=10, fontweight='bold', color='#283593')
ax.annotate('', xy=(14.5, 8.5), xytext=(11.5, 8.5),
            arrowprops=dict(arrowstyle='<->', color='#283593', lw=1.5, ls='--'))

plt.tight_layout()
plt.show()

print("Three Serving Components:")
print("1. Harmful Content Detection Service: Predicts harm probability per category")
print("2. Violation Enforcement Service: Removes post + notifies user (HIGH confidence)")
print("3. Demoting Service: Reduces visibility + queues for human review (LOW confidence)")

---

## 7. Simulating the End-to-End Pipeline

Let's simulate what happens when a post enters the system. Think of it like watching a package go through airport security -- each station checks something different.

In [ ]:
import numpy as np

class HarmfulContentDetectionPipeline:
    """
    Simulates the complete harmful content detection pipeline.
    
    Think of it like an airport security system:
    1. Your bag (post) goes through the X-ray scanner (feature extraction)
    2. The scanner analyzes it for different threats (multi-task classification)
    3. Based on the results:
       - Clear threat? --> Confiscated immediately (violation enforcement)
       - Suspicious? --> Set aside for manual inspection (demoting service)
       - Clean? --> You go on your way (post goes live)
    """
    
    def __init__(self, harm_categories, thresholds):
        self.harm_categories = harm_categories
        self.remove_thresholds = thresholds['remove']  # Above this: remove
        self.demote_thresholds = thresholds['demote']  # Above this: demote
    
    def extract_features(self, post):
        """Simulate feature extraction from a post."""
        features = {
            'text_embedding': np.random.randn(768).tolist()[:5],  # DistilBERT: 768-dim
            'image_embedding': np.random.randn(512).tolist()[:5],  # CLIP: 512-dim
            'author_violations': post.get('author_violations', 0),
            'num_reports': post.get('num_reports', 0),
            'account_age_days': post.get('account_age_days', 365),
        }
        return features
    
    def predict(self, post):
        """Simulate multi-task model prediction."""
        # In reality, this would be a neural network forward pass
        # Here we simulate based on textual cues for demonstration
        text = post.get('text', '').lower()
        
        probs = {}
        for cat in self.harm_categories:
            base_prob = np.random.uniform(0.01, 0.1)  # Low base probability
            
            # Boost based on content signals (simplified simulation)
            if cat == 'violence' and any(w in text for w in ['fight', 'kill', 'attack', 'violent']):
                base_prob += np.random.uniform(0.5, 0.9)
            elif cat == 'hate_speech' and any(w in text for w in ['hate', 'inferior', 'disgusting']):
                base_prob += np.random.uniform(0.5, 0.9)
            elif cat == 'nudity' and any(w in text for w in ['nude', 'explicit', 'nsfw']):
                base_prob += np.random.uniform(0.5, 0.9)
            
            # Author risk factor
            if post.get('author_violations', 0) > 2:
                base_prob += 0.15
            
            probs[cat] = min(base_prob, 0.99)
        
        return probs
    
    def decide_action(self, predictions):
        """Determine action based on prediction confidence."""
        for cat, prob in predictions.items():
            if prob >= self.remove_thresholds.get(cat, 0.8):
                return 'REMOVE', cat, prob
        
        for cat, prob in predictions.items():
            if prob >= self.demote_thresholds.get(cat, 0.4):
                return 'DEMOTE', cat, prob
        
        return 'ALLOW', None, 0.0
    
    def process_post(self, post):
        """Full pipeline: extract features -> predict -> decide."""
        features = self.extract_features(post)
        predictions = self.predict(post)
        action, reason, confidence = self.decide_action(predictions)
        
        return {
            'post_text': post.get('text', '(no text)'),
            'predictions': predictions,
            'action': action,
            'reason': reason,
            'confidence': confidence,
        }


# Set up the pipeline
np.random.seed(42)

pipeline = HarmfulContentDetectionPipeline(
    harm_categories=['violence', 'nudity', 'hate_speech', 'self_harm', 'harassment'],
    thresholds={
        'remove': {'violence': 0.8, 'nudity': 0.85, 'hate_speech': 0.8, 'self_harm': 0.7, 'harassment': 0.85},
        'demote': {'violence': 0.4, 'nudity': 0.5, 'hate_speech': 0.4, 'self_harm': 0.3, 'harassment': 0.5},
    }
)

# Test posts
test_posts = [
    {'text': 'Just had an amazing dinner with friends!', 'author_violations': 0, 'account_age_days': 1000},
    {'text': 'Watch this brutal fight video - very violent!', 'author_violations': 3, 'account_age_days': 15},
    {'text': 'I hate everyone in this group, they are all inferior', 'author_violations': 2, 'account_age_days': 60},
    {'text': 'Beautiful sunset over the mountains', 'author_violations': 0, 'account_age_days': 2000},
]

print("=" * 80)
print("HARMFUL CONTENT DETECTION PIPELINE SIMULATION")
print("=" * 80)

for i, post in enumerate(test_posts, 1):
    result = pipeline.process_post(post)
    print(f"\n--- Post {i} ---")
    print(f"  Text: '{result['post_text']}'")
    print(f"  Predictions:")
    for cat, prob in result['predictions'].items():
        bar = '#' * int(prob * 30)
        print(f"    {cat:15s}: {prob:.3f} |{bar}")
    
    action_color = {'REMOVE': '!!!', 'DEMOTE': '???', 'ALLOW': '>>>'}
    print(f"  Action: {action_color[result['action']]} {result['action']}")
    if result['reason']:
        print(f"  Reason: {result['reason']} (confidence: {result['confidence']:.3f})")

---

## 8. Interview Cheat Sheet

### The 8-Step Framework

```
1. CLARIFY:   Multi-modal posts, multiple harm categories, real-time + batch, multilingual
2. FRAME:     Multi-task classification with early fusion of modalities
3. DATA:      Users + Posts + Interactions; natural labels (reports) + hand labels (evaluation)
4. FEATURES:  Text (DistilBERT) + Image (CLIP/SimCLR) + Reactions + Author + Context
5. MODEL:     Neural network with shared layers + task-specific classification heads
6. TRAINING:  Cross-entropy per task, sum of losses, gradient blending, focal loss
7. METRICS:   Offline = PR-AUC + ROC-AUC; Online = Harmful impressions, Valid appeals, Proactive rate
8. SERVING:   Detection service --> Violation enforcement (high conf) or Demoting (low conf)
```

### Key Phrases to Drop in Interview

| When Asked About... | Say This... |
|---------------------|-------------|
| Why early fusion? | "Memes and combined content can be harmful even when each modality is benign in isolation. Early fusion captures cross-modal interactions that late fusion misses." |
| Why multi-task? | "Shared layers enable knowledge transfer between tasks. Training data for violence also helps the shared layers learn patterns useful for hate speech. Plus, one model to maintain." |
| Class imbalance? | "Focal loss to focus on hard examples, oversampling rare categories, and strategic threshold tuning per harm category." |
| Adversarial content? | "Obfuscation detection, character normalization, image perturbation augmentation during training." |
| Confidence threshold? | "Tune per category. Violence has a low threshold (better safe). Satire/controversy has higher thresholds to avoid over-removal." |
| Appeals? | "Track valid appeal rate as an online metric. High appeal rate signals the model is too aggressive." |
| New harm types? | "Active learning pipeline: surface uncertain predictions for human review, retrain periodically with new labeled data." |
| Latency? | "Feature extraction at post creation time. Model inference is a single fast forward pass. Video processed asynchronously." |

### Common Follow-Up Questions

| Question | Strong Answer |
|----------|---------------|
| How do you handle multilingual content? | DistilmBERT produces similar embeddings for same-meaning text across languages |
| What about encrypted or private content? | Only scan public/semi-public posts; E2E encrypted messages are out of scope |
| How do you prevent bias? | Regular fairness audits per demographic group, diverse annotator pools |
| Scale considerations? | Feature extraction is the bottleneck; use GPU inference clusters; batch processing for non-urgent categories |

In [ ]:
# Final summary: The complete picture
print("=" * 70)
print("HARMFUL CONTENT DETECTION - SYSTEM DESIGN SUMMARY")
print("=" * 70)
print()
print("PROBLEM:  Detect harmful content in social media posts")
print("SCALE:    500M posts/day, multiple languages")
print("INPUT:    Post (text + image + video + metadata)")
print("OUTPUT:   Per-category harm probabilities")
print()
print("KEY DECISIONS:")
print("  1. Early Fusion (not late) --> captures cross-modal harm (memes)")
print("  2. Multi-Task (not single/multi-label) --> shared + specialized")
print("  3. Neural Network backbone --> handles multi-task naturally")
print("  4. DistilBERT for text (not BERT) --> faster, multilingual")
print("  5. CLIP/SimCLR for images --> pre-trained visual features")
print("  6. Natural labels for training --> user reports (free, scalable)")
print("  7. Hand labels for evaluation --> accurate ground truth")
print()
print("SERVING:")
print("  High confidence harmful --> REMOVE immediately")
print("  Low confidence harmful  --> DEMOTE + human review")
print("  Not harmful             --> Allow through")
print()
print("NEXT NOTEBOOKS:")
print("  02: Deep dive into multimodal fusion and feature engineering")
print("  03: Multi-task training, loss functions, and evaluation metrics")

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)